# Entrenamiento Age Classifier — MobileNetV2
**Pasos:** instalar deps → descargar dataset → entrenar → descargar modelo

In [ ]:
# Verificar GPU disponible
import tensorflow as tf
print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
!pip install -q kaggle scikit-learn matplotlib opencv-python-headless

## Paso 1 — Dataset de Kaggle
Ve a https://www.kaggle.com/settings → API → *Create New Token* → descarga `kaggle.json`
Luego sube ese fichero con la celda de abajo.

In [ ]:
from google.colab import files
uploaded = files.upload()   # sube kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d frabbisw/facial-age -p /content --unzip
import os, pathlib
# El dataset se descomprime en una subcarpeta — encontrarla automáticamente
dataset_root = None
for p in pathlib.Path('/content').iterdir():
    if p.is_dir() and any(p.iterdir()):
        sub = list(p.iterdir())[0]
        if sub.is_dir() and sub.name.lstrip('0').isdigit() if sub.name.lstrip('0') else True:
            dataset_root = str(p)
            break
if not dataset_root:
    dataset_root = '/content'
print('Dataset en:', dataset_root)
print('Carpetas de edad encontradas:', len([x for x in pathlib.Path(dataset_root).iterdir() if x.is_dir()]))

## Paso 2 — Entrenamiento

In [ ]:
import json, logging, os, sys
from pathlib import Path
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import callbacks, layers, models
from tensorflow.keras.applications import MobileNetV2

DATASET_DIR     = Path(dataset_root)
OUTPUT_DIR      = Path('/content/model')
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_PATH      = OUTPUT_DIR / 'age_classifier.keras'

IMG_SIZE        = (200, 200)
BATCH_SIZE      = 64       # más grande en GPU
EPOCHS_HEAD     = 20
EPOCHS_FINETUNE = 15
LR_HEAD         = 1e-3
LR_FINETUNE     = 1e-5
MINOR_THRESHOLD = 18
VAL_SPLIT       = 0.2
SEED            = 42
MAX_IMGS_PER_AGE= 250

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')
log = logging.getLogger('train')
print(f'TF {tf.__version__} | GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
def load_paths_and_labels(dataset_dir):
    filepaths, labels = [], []
    counts = {0:0, 1:0}
    rng = np.random.default_rng(SEED)
    for age_dir in sorted(dataset_dir.iterdir()):
        if not age_dir.is_dir(): continue
        name = age_dir.name.lstrip('0') or '0'
        if not name.isdigit(): continue   # ignorar carpetas como .config
        age  = int(name)
        label = 1 if age < MINOR_THRESHOLD else 0
        imgs  = list(age_dir.glob('*.png'))
        if len(imgs) > MAX_IMGS_PER_AGE:
            imgs = [imgs[i] for i in rng.choice(len(imgs), MAX_IMGS_PER_AGE, replace=False)]
        for p in imgs:
            filepaths.append(str(p)); labels.append(label); counts[label]+=1
    print(f'Total: {len(filepaths)} | menores(1): {counts[1]} | adultos(0): {counts[0]}')
    return np.array(filepaths), np.array(labels, dtype=np.int32)

def preprocess(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    return tf.keras.applications.mobilenet_v2.preprocess_input(img)

def augment(img):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    return tf.clip_by_value(img, -1.0, 1.0)

def make_ds(paths, labels, aug=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle: ds = ds.shuffle(len(paths), seed=SEED)
    ds = ds.map(lambda p,l: (preprocess(p), l), num_parallel_calls=tf.data.AUTOTUNE)
    if aug: ds = ds.map(lambda x,l: (augment(x), l), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

def build_model(trainable_base=False):
    base = MobileNetV2(input_shape=(*IMG_SIZE,3), include_top=False, weights='imagenet')
    base.trainable = trainable_base
    inp = tf.keras.Input(shape=(*IMG_SIZE,3))
    x   = base(inp, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dense(256, activation='relu')(x)
    x   = layers.Dropout(0.4)(x)
    x   = layers.Dense(64, activation='relu')(x)
    x   = layers.Dropout(0.2)(x)
    out = layers.Dense(1, activation='sigmoid', name='minor_prob')(x)
    return models.Model(inp, out, name='age_classifier')

all_paths, all_labels = load_paths_and_labels(DATASET_DIR)
tr_p, val_p, tr_l, val_l = train_test_split(all_paths, all_labels,
    test_size=VAL_SPLIT, stratify=all_labels, random_state=SEED)
print(f'Train: {len(tr_p)} | Val: {len(val_p)}')
train_ds = make_ds(tr_p, tr_l, aug=True,  shuffle=True)
val_ds   = make_ds(val_p, val_l, aug=False, shuffle=False)

In [ ]:
# ── FASE 1: cabeza ──
print('='*50)
print('FASE 1 — Entrenamiento de cabeza')
print('='*50)
model = build_model(trainable_base=False)
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_HEAD),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.Precision(name='precision')]
)
h1 = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
    class_weight={0:1.0, 1:1.0},
    callbacks=[
        callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
    ], verbose=1
)

In [ ]:
# ── FASE 2: fine-tuning ──
print('='*50)
print('FASE 2 — Fine-tuning')
print('='*50)
base = model.layers[1]
base.trainable = True
for layer in base.layers[:-40]: layer.trainable = False
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_FINETUNE),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.Precision(name='precision')]
)
h2 = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS_FINETUNE,
    class_weight={0:1.0, 1:1.0},
    callbacks=[
        callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=6, restore_best_weights=True, verbose=1),
        callbacks.ModelCheckpoint(str(MODEL_PATH), monitor='val_auc', mode='max', save_best_only=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, verbose=1),
    ], verbose=1
)
if not MODEL_PATH.exists(): model.save(MODEL_PATH)
print('Modelo guardado en:', MODEL_PATH)

In [ ]:
# ── Evaluación final ──
best = tf.keras.models.load_model(MODEL_PATH)
probs = best.predict(val_ds, verbose=0).ravel()
preds = (probs >= 0.5).astype(int)
print(f'AUC-ROC: {roc_auc_score(val_l, probs):.4f}')
print(classification_report(val_l, preds, target_names=['adulto(0)','menor(1)']))
print('Matriz de confusión:')
print(confusion_matrix(val_l, preds))

In [ ]:
# ── Descargar modelo ──
from google.colab import files
files.download(str(MODEL_PATH))
print('Descarga iniciada: age_classifier.keras')
print('Guárdalo en: age-detection/model/age_classifier.keras')